# **Muhammad Ibtisam Afzal**

# Assignment No. 10 — Advanced NLP Techniques and Feature Engineering

**Objective:** Apply advanced NLP techniques and feature engineering to three short news-style text documents. This notebook answers all eight questions in the assignment.

In [1]:
# 1. Imports and NLTK resource setup
import sys, subprocess, importlib.util
if importlib.util.find_spec("nltk") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk", "-q"])
import nltk, pandas as pd, numpy as np
from IPython.display import display

resources = [
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
    ("taggers/averaged_perceptron_tagger_eng", "averaged_perceptron_tagger_eng"),
    ("chunkers/maxent_ne_chunker_tab", "maxent_ne_chunker_tab"),
    ("corpora/words", "words"),
    ("corpora/wordnet", "wordnet"),
    ("corpora/omw-1.4", "omw-1.4"),
]
for path, package in resources:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(package, quiet=True)
print("NLTK version:", nltk.__version__)

NLTK version: 3.9.1


## 2. Text-document collection\n\nThree short news-style documents are used to demonstrate the techniques.

In [2]:
documents = [
 {"doc_id":"D1","topic":"Technology","text":"OpenAI announced a new artificial intelligence research center in London. The team will develop safer language models and help universities study machine learning applications."},
 {"doc_id":"D2","topic":"Public Health","text":"The World Health Organization reported that hospitals in Karachi are expanding vaccination programs. Doctors said that community education can reduce infectious diseases and protect children."},
 {"doc_id":"D3","topic":"Environment","text":"Researchers at the University of Oxford are studying renewable energy projects. The scientists believe solar power and wind farms can reduce air pollution and support sustainable cities."}
]
df = pd.DataFrame(documents)
display(df)
texts = df["text"].tolist()

,doc_id,topic,text
0,D1,Technology,OpenAI announced a new artificial intelligence...
1,D2,Public Health,The World Health Organization reported that ho...
2,D3,Environment,Researchers at the University of Oxford are st...


## 3. POS tagging\n\nPart-of-Speech tagging assigns grammatical categories (such as noun, verb, adjective, and adverb) to tokens.

In [3]:
from nltk import word_tokenize, pos_tag
pos_results = {}
for row in documents:
    tagged = pos_tag(word_tokenize(row["text"]))
    pos_results[row["doc_id"]] = tagged
    print("\\n", row["doc_id"], "—", row["topic"])
    print(tagged)
display(pd.DataFrame(pos_results["D1"], columns=["Token","POS tag"]))

\n D1 — Technology
[('OpenAI', 'NNP'), ('announced', 'VBD'), ('a', 'DT'), ('new', 'JJ'), ('artificial', 'JJ'), ('intelligence', 'NN'), ('research', 'NN'), ('center', 'NN'), ('in', 'IN'), ('London', 'NNP'), ('.', '.'), ('The', 'DT'), ('team', 'NN'), ('will', 'MD'), ('develop', 'VB'), ('safer', 'JJR'), ('language', 'NN'), ('models', 'NNS'), ('and', 'CC'), ('help', 'VB'), ('universities', 'NNS'), ('study', 'VB'), ('machine', 'NN'), ('learning', 'VBG'), ('applications', 'NNS'), ('.', '.')]
\n D2 — Public Health
[('The', 'DT'), ('World', 'NNP'), ('Health', 'NNP'), ('Organization', 'NNP'), ('reported', 'VBD'), ('that', 'DT'), ('hospitals', 'NNS'), ('in', 'IN'), ('Karachi', 'NNP'), ('are', 'VBP'), ('expanding', 'VBG'), ('vaccination', 'NN'), ('programs', 'NNS'), ('.', '.'), ('Doctors', 'NNS'), ('said', 'VBD'), ('that', 'IN'), ('community', 'NN'), ('education', 'NN'), ('can', 'MD'), ('reduce', 'VB'), ('infectious', 'JJ'), ('diseases', 'NNS'), ('and', 'CC'), ('protect', 'JJ'), ('children', 'NNS

,Token,POS tag
0,OpenAI,NNP
1,announced,VBD
2,a,DT
3,new,JJ
4,artificial,JJ
5,intelligence,NN
6,research,NN
7,center,NN
8,in,IN
9,London,NNP


**Interpretation:** POS tags describe the grammatical role of each token in context. For example, `NN`/`NNS` denote nouns, `VB`-family tags denote verbs, and `JJ`-family tags denote adjectives.

## 4. Named Entity Recognition (NER)\n\nNER identifies named entities and categorizes them, for example as organizations, people, geopolitical entities, or locations.

In [4]:
from nltk import ne_chunk
ner_results = {}
def extract_entities(tree):
    found = []
    for node in tree:
        if hasattr(node, "label"):
            found.append((" ".join(t for t, tag in node.leaves()), node.label()))
    return found

ner_rows = []
for row in documents:
    tree = ne_chunk(pos_tag(word_tokenize(row["text"])), binary=False)
    ner_results[row["doc_id"]] = tree
    print("\\n", row["doc_id"], "NER tree:")
    print(tree)
    for entity, label in extract_entities(tree):
        ner_rows.append({"Document":row["doc_id"],"Entity":entity,"NER label":label})
display(pd.DataFrame(ner_rows, columns=["Document","Entity","NER label"]))

\n D1 NER tree:
(S
  (ORGANIZATION OpenAI/NNP)
  announced/VBD
  a/DT
  new/JJ
  artificial/JJ
  intelligence/NN
  research/NN
  center/NN
  in/IN
  (GPE London/NNP)
  ./.
  The/DT
  team/NN
  will/MD
  develop/VB
  safer/JJR
  language/NN
  models/NNS
  and/CC
  help/VB
  universities/NNS
  study/VB
  machine/NN
  learning/VBG
  applications/NNS
  ./.)
\n D2 NER tree:
(S
  The/DT
  (ORGANIZATION World/NNP)
  Health/NNP
  Organization/NNP
  reported/VBD
  that/DT
  hospitals/NNS
  in/IN
  (GPE Karachi/NNP)
  are/VBP
  expanding/VBG
  vaccination/NN
  programs/NNS
  ./.
  (PERSON Doctors/NNS)
  said/VBD
  that/IN
  community/NN
  education/NN
  can/MD
  reduce/VB
  infectious/JJ
  diseases/NNS
  and/CC
  protect/JJ
  children/NNS
  ./.)
\n D3 NER tree:
(S
  Researchers/NNS
  at/IN
  the/DT
  (ORGANIZATION University/NNP)
  of/IN
  (GPE Oxford/NNP)
  are/VBP
  studying/VBG
  renewable/JJ
  energy/NN
  projects/NNS
  ./.
  The/DT
  scientists/NNS
  believe/VBP
  solar/JJ
  power/NN
  and/

,Document,Entity,NER label
0,D1,OpenAI,ORGANIZATION
1,D1,London,GPE
2,D2,World,ORGANIZATION
3,D2,Karachi,GPE
4,D2,Doctors,PERSON
5,D3,University,ORGANIZATION
6,D3,Oxford,GPE


**Interpretation:** NER detects names and assigns entity types. Results depend on the trained model and sentence context; not every proper noun is guaranteed to be recognized correctly.

## 5. Chunking and chinking\n\nChunking groups tokens matching a grammatical pattern. Chinking removes selected tokens from within a chunk. The grammar below extracts noun phrases, then removes verbs and prepositions.

In [5]:
from nltk import RegexpParser
grammar = r'''
NP: {<DT>?<JJ>*<NN.*>+}
    }<VB.*|IN>{
'''
parser = RegexpParser(grammar)
for row in documents:
    tagged = pos_tag(word_tokenize(row["text"]))
    parsed = parser.parse(tagged)
    print("\\n", row["doc_id"], "chunked/chinked parse:")
    print(parsed)

\n D1 chunked/chinked parse:
(S
  (NP OpenAI/NNP)
  announced/VBD
  (NP
    a/DT
    new/JJ
    artificial/JJ
    intelligence/NN
    research/NN
    center/NN)
  in/IN
  (NP London/NNP)
  ./.
  (NP The/DT team/NN)
  will/MD
  develop/VB
  safer/JJR
  (NP language/NN models/NNS)
  and/CC
  help/VB
  (NP universities/NNS)
  study/VB
  (NP machine/NN)
  learning/VBG
  (NP applications/NNS)
  ./.)
\n D2 chunked/chinked parse:
(S
  (NP The/DT World/NNP Health/NNP Organization/NNP)
  reported/VBD
  (NP that/DT hospitals/NNS)
  in/IN
  (NP Karachi/NNP)
  are/VBP
  expanding/VBG
  (NP vaccination/NN programs/NNS)
  ./.
  (NP Doctors/NNS)
  said/VBD
  that/IN
  (NP community/NN education/NN)
  can/MD
  reduce/VB
  (NP infectious/JJ diseases/NNS)
  and/CC
  (NP protect/JJ children/NNS)
  ./.)
\n D3 chunked/chinked parse:
(S
  (NP Researchers/NNS)
  at/IN
  (NP the/DT University/NNP)
  of/IN
  (NP Oxford/NNP)
  are/VBP
  studying/VBG
  (NP renewable/JJ energy/NN projects/NNS)
  ./.
  (NP The/DT 

**Interpretation:** Chunking identifies shallow phrase structures without producing a complete syntactic parse. Chinking removes patterns that should not remain inside the chunks.

## 6. Lemmatization compared with stemming\n\nStemming truncates words using rules and may produce non-dictionary forms. Lemmatization uses vocabulary and morphology to return a dictionary base form.

In [6]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
stemmer, lemmatizer = PorterStemmer(), WordNetLemmatizer()
samples = {
 "studies":wordnet.NOUN, "studying":wordnet.VERB, "announced":wordnet.VERB,
 "models":wordnet.NOUN, "children":wordnet.NOUN, "better":wordnet.ADJ, "running":wordnet.VERB
}
comparison = [{"Original word":w, "Porter stem":stemmer.stem(w),
               "WordNet lemma":lemmatizer.lemmatize(w, pos=p)} for w,p in samples.items()]
display(pd.DataFrame(comparison))

,Original word,Porter stem,WordNet lemma
0,studies,studi,study
1,studying,studi,study
2,announced,announc,announce
3,models,model,model
4,children,children,child
5,better,better,good
6,running,run,run


**Comparison:** Stemming is fast but can return truncated forms that are not valid words. Lemmatization is linguistically informed and often returns readable dictionary forms, but its quality depends on the correct part-of-speech input.

## 7. WordNet synonyms and antonyms for five words\n\nWordNet is a lexical database that groups words into synsets. The following code retrieves up to eight unique examples of synonyms and antonyms for each selected word.

In [7]:
from nltk.corpus import wordnet as wn
selected_words = ["safe", "new", "reduce", "strong", "clean"]
rows = []
for word in selected_words:
    synonyms, antonyms = set(), set()
    for syn in wn.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().replace("_"," "))
            for ant in lemma.antonyms():
                antonyms.add(ant.name().replace("_"," "))
    synonyms.discard(word)
    rows.append({"Word":word,
                 "Synonyms (sample)":", ".join(sorted(synonyms)[:8]) or "Not found",
                 "Antonyms (sample)":", ".join(sorted(antonyms)[:8]) or "Not found"})
display(pd.DataFrame(rows))

,Word,Synonyms (sample),Antonyms (sample)
0,safe,"condom, dependable, good, prophylactic, rubber...","dangerous, out"
1,new,"Modern, New, fresh, freshly, newfangled, newly...","old, worn"
2,reduce,"abbreviate, abridge, boil down, bring down, co...","blow up, expand, gain, oxidise, oxidize"
3,strong,"firm, hard, impregnable, inviolable, potent, s...","impotent, weak"
4,clean,"blank, clean and jerk, clean house, clean-livi...","dirty, unclean, unfairly"


**Interpretation:** A word can have multiple senses. Therefore, WordNet may return alternatives that are not suitable in every context; these are lexical candidates, not guaranteed replacements.

## 8. Bag of Words (BoW) numerical features\n\nBag of Words converts text into a document-term matrix. Each column is a vocabulary term and each cell contains its frequency in that document.

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
bow_vectorizer = CountVectorizer(stop_words="english")
bow_matrix = bow_vectorizer.fit_transform(texts)
bow_df = pd.DataFrame(bow_matrix.toarray(), index=df["doc_id"],
                      columns=bow_vectorizer.get_feature_names_out())
display(bow_df)
print("Matrix shape (documents, terms):", bow_matrix.shape)

,air,announced,applications,artificial,believe,center,children,cities,community,develop,...,study,studying,support,sustainable,team,universities,university,vaccination,wind,world
doc_id,,,,,,,,,,,,,,,,,,,,,
D1,0,1,1,1,0,1,0,0,0,1,...,1,0,0,0,1,1,0,0,0,0
D2,0,0,0,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,1,0,1
D3,1,0,0,0,1,0,0,1,0,0,...,0,1,1,1,0,0,1,0,1,0


Matrix shape (documents, terms): (3, 55)


**Interpretation:** BoW provides numerical features for machine-learning models but ignores word order and semantic relationships. Stop-word removal reduces the influence of common words.

## 9. Feature selection and feature extraction

Two approaches are demonstrated:

- **Feature selection:** Chi-square (`SelectKBest`) selects original terms associated with the topic labels.
- **Feature extraction:** Truncated SVD transforms the TF-IDF matrix into a smaller set of latent numerical dimensions.

The labels are illustrative. With only three documents, these outputs demonstrate the methods and are not statistically generalizable.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import TruncatedSVD

labels = df["topic"].to_numpy()
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
terms = tfidf_vectorizer.get_feature_names_out()

k = min(8, len(terms))
selector = SelectKBest(score_func=chi2, k=k)
selected = selector.fit_transform(tfidf_matrix, labels)
scores = pd.DataFrame({"Feature":terms, "Chi-square score":selector.scores_,
                       "Selected":selector.get_support()}).sort_values("Chi-square score", ascending=False)
print("Selected terms:", terms[selector.get_support()].tolist())
display(scores)

n_components = min(2, tfidf_matrix.shape[0]-1, tfidf_matrix.shape[1]-1)
if n_components >= 1:
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    latent = svd.fit_transform(tfidf_matrix)
    display(pd.DataFrame(latent, index=df["doc_id"],
                         columns=[f"Latent feature {i+1}" for i in range(n_components)]))
    print("Explained variance ratio:", svd.explained_variance_ratio_)
else:
    print("SVD skipped: insufficient matrix dimensions.")

Selected terms: ['karachi', 'organization', 'programs', 'protect', 'reported', 'said', 'vaccination', 'world']


,Feature,Chi-square score,Selected
6,children,0.477024,False
21,karachi,0.477024,True
14,expanding,0.477024,False
11,doctors,0.477024,False
12,education,0.477024,False
10,diseases,0.477024,False
8,community,0.477024,False
19,infectious,0.477024,False
16,health,0.477024,False
18,hospitals,0.477024,False


,Latent feature 1,Latent feature 2
doc_id,,
D1,-3.591546e-17,1.000000e+00
D2,7.183336e-01,1.152618e-16
D3,7.183336e-01,-2.870083e-16


Explained variance ratio: [0.17385585 0.33692793]


**Interpretation:** Feature selection retains a subset of the original terms, so their meanings remain directly interpretable. Feature extraction creates new dimensions that combine information from original terms. The very small sample limits the reliability of both.

## 10. Document similarity between two documents\n\nCosine similarity compares the angle between TF-IDF vectors. Values closer to 1 indicate more similar term-weight patterns; values closer to 0 indicate less lexical overlap.

In [10]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(tfidf_matrix)
print(f"Cosine similarity between D1 ({df.loc[0,'topic']}) and D2 ({df.loc[1,'topic']}): {similarity[0,1]:.4f}")
display(pd.DataFrame(similarity, index=df["doc_id"], columns=df["doc_id"]))

Cosine similarity between D1 (Technology) and D2 (Public Health): 0.0000


doc_id,D1,D2,D3
doc_id,,,
D1,1.0,0.000000,0.000000
D2,0.0,1.000000,0.032006
D3,0.0,0.032006,1.000000


**Interpretation:** The D1–D2 value measures overlap in their TF-IDF-weighted vocabulary. Because the documents discuss different topics, a lower value is expected. This is lexical similarity, not a complete measure of semantic meaning.

## 11. Conclusion

All eight assignment requirements are covered:

1. POS tagging assigns grammatical labels to tokens.
2. NER identifies and categorizes named entities.
3. Chunking and chinking identify phrase chunks and remove selected patterns.
4. Lemmatization and stemming are compared on sample words.
5. WordNet synonyms and antonyms are retrieved for five words.
6. Bag of Words converts documents into a numerical term-frequency matrix.
7. Chi-square feature selection and SVD feature extraction are demonstrated.
8. Cosine similarity between two documents is calculated using TF-IDF.

**Note:** This small dataset is suitable for demonstrating techniques. Real-world NLP research requires a larger, representative corpus and appropriate validation.